# Notebook 5: Data Integration

## Objective

The purpose of this notebook is to combine the historical stock price dataset with the company fundamentals dataset into a single analysis-ready table.

The historical stock price data contains daily observations for each company, while the company fundamentals dataset contains static financial metrics collected from Yahoo Finance.

Rather than merging the datasets using pandas, this notebook demonstrates how SQL can be used for data integration. SQL is commonly used by data analysts to combine tables stored in relational databases using shared keys.

The resulting dataset will contain both daily stock market information and company-level financial metrics, providing a richer dataset for future analysis and dashboard development.

---

## Business Question

**How can historical stock prices and company financial metrics be combined into a single dataset to support more comprehensive investment analysis?**

In [29]:
import pandas as pd
import sqlite3

prices = pd.read_csv("../data/processed/master_stock_data.csv")

fundamentals = pd.read_csv("../data/processed/company_fundamentals.csv")

conn = sqlite3.connect("stock_database.db")

In [30]:
prices.to_sql(
    "stock_prices",
    conn,
    if_exists="replace",
    index=False
)

fundamentals.to_sql(
    "company_info",
    conn,
    if_exists="replace",
    index=False
)

5

In [31]:
query = """

SELECT
    sp.*,
    ci.Company,
    ci.Sector,
    ci.`Market Cap`,
    ci.`Forward PE`,
    ci.EPS,
    ci.Revenue,
    ci.`Net Income`,
    ci.`Profit Margin`,
    ci.`Return on Equity`
FROM stock_prices sp

LEFT JOIN company_info ci

ON sp.Ticker = ci.Ticker

"""


In [32]:
merged = pd.read_sql(query, conn)

In [33]:
merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 6290 entries, 0 to 6289
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Date              6290 non-null   str    
 1   Adj Close         6290 non-null   float64
 2   Close             6290 non-null   float64
 3   High              6290 non-null   float64
 4   Low               6290 non-null   float64
 5   Open              6290 non-null   float64
 6   Volume            6290 non-null   int64  
 7   Ticker            6290 non-null   str    
 8   Company           6290 non-null   str    
 9   Sector            6290 non-null   str    
 10  Market Cap        6290 non-null   int64  
 11  Forward PE        6290 non-null   float64
 12  EPS               6290 non-null   float64
 13  Revenue           6290 non-null   int64  
 14  Net Income        6290 non-null   int64  
 15  Profit Margin     6290 non-null   float64
 16  Return on Equity  6290 non-null   float64
dtypes: flo

In [34]:
merged.describe()

,Adj Close,Close,High,Low,Open,Volume,Market Cap,Forward PE,EPS,Revenue,Net Income,Profit Margin,Return on Equity
count,6290.000000,6290.000000,6290.000000,6290.000000,6290.000000,6.290000e+03,6.290000e+03,6290.000000,6290.000000,6.290000e+03,6.290000e+03,6290.000000,6290.000000
mean,148.054515,150.541544,152.188571,148.772205,150.486956,1.335117e+08,3.699387e+12,22.445711,10.460000,4.376960e+11,1.316820e+11,0.359206,0.705886
std,93.522212,95.856497,96.588966,95.044535,95.854300,1.815927e+08,8.705534e+11,4.843081,3.898104,1.683506e+11,2.604097e+10,0.166481,0.477967
min,4.885130,4.910000,5.248500,4.517000,5.002500,7.164500e+06,2.583212e+12,15.274478,6.530000,2.534910e+11,9.079800e+10,0.122240,0.242850
25%,91.879398,92.422499,93.963749,90.855000,92.474001,3.008338e+07,2.737898e+12,19.029018,7.590000,3.182730e+11,1.225750e+11,0.271520,0.340140
50%,139.895920,141.529999,143.254997,139.764999,141.559998,5.415195e+07,4.138015e+12,24.300047,8.250000,4.224980e+11,1.252160e+11,0.379190,0.388850
75%,182.509521,184.224998,185.882248,182.474998,184.152500,1.223203e+08,4.315440e+12,24.304340,13.120000,4.514420e+11,1.596130e+11,0.393420,1.142880
max,460.325623,467.559998,468.350006,464.459991,467.000000,1.543911e+09,4.722368e+12,29.320671,16.810000,7.427760e+11,1.602080e+11,0.629660,1.414710


In [35]:
merged.head()


,Date,Adj Close,Close,High,Low,Open,Volume,Ticker,Company,Sector,Market Cap,Forward PE,EPS,Revenue,Net Income,Profit Margin,Return on Equity
0,2020-01-02,72.333878,75.087502,75.150002,73.797501,74.059998,135480400,AAPL,Apple Inc.,Technology,4138015391744,29.320671,8.25,451442016256,122575003648,0.27152,1.41471
1,2020-01-03,71.630661,74.357498,75.144997,74.125000,74.287498,146322800,AAPL,Apple Inc.,Technology,4138015391744,29.320671,8.25,451442016256,122575003648,0.27152,1.41471
2,2020-01-06,72.201416,74.949997,74.989998,73.187500,73.447502,118387200,AAPL,Apple Inc.,Technology,4138015391744,29.320671,8.25,451442016256,122575003648,0.27152,1.41471
3,2020-01-07,71.861855,74.597504,75.224998,74.370003,74.959999,108872000,AAPL,Apple Inc.,Technology,4138015391744,29.320671,8.25,451442016256,122575003648,0.27152,1.41471
4,2020-01-08,73.017845,75.797501,76.110001,74.290001,74.290001,132079200,AAPL,Apple Inc.,Technology,4138015391744,29.320671,8.25,451442016256,122575003648,0.27152,1.41471


In [36]:
merged.to_csv(
    "../data/processed/stock_data_complete.csv",
    index=False
)

In [37]:
conn.close()

## Summary

The historical stock price dataset and company fundamentals dataset were successfully integrated using SQL.

A LEFT JOIN was performed on the shared `Ticker` column, ensuring that every historical stock price observation was retained while enriching each record with company-level financial metrics such as market capitalization, valuation ratios, profitability, and revenue.

The final merged dataset provides a comprehensive view of each company's daily market performance alongside its underlying financial characteristics. This dataset will serve as the primary data source for subsequent exploratory analysis, feature engineering, and interactive dashboard development.